<a href="https://colab.research.google.com/github/Oluwatimileyingit/DecodeLabs-Internship-/blob/sales-eda-project/Sales_EDA_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sales EDA

## Sales Data Exploratory Analysis

## Introduction

This project explores a sales dataset containing 1,200 records and 14 variables. The analysis uses Python to investigate product sales performance, revenue distribution, quantity sold, order status and data quality.

The project applies exploratory data analysis (EDA) techniques to summarise the data, identify patterns, visualise key metrics and assess potential limitations. Results are compared with an earlier Excel analysis to check consistency across tools.

## Objectives

- Assess the completeness and quality of the dataset.
- Summarise revenue and quantity sold by product.
- Compare average revenue per unit across products.
- Examine order status, payment methods and referral sources.
- Communicate findings through tables, visualisations and concise interpretations.

## Tools

- Python
- Pandas and NumPy
- Matplotlib and Seaborn
- Jupyter Notebook

In [1]:
##Importing Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Data Loading and Initial Inspection

The sales dataset was imported into Python using the Pandas library. Initial inspection was carried out to understand the structure, size, and contents of the dataset before proceeding with the analysis.

The dataset contains **1,200 rows and 14 columns**, representing sales records and related information, including order dates, products, quantities, payment methods, order statuses, unit prices, and total prices.

Functions such as `head()`, `tail()`, `shape`, `columns`, and `info()` were used to examine sample records, identify column names, determine data types, and assess the overall structure of the dataset.

This initial inspection provided an overview of the available variables and helped identify areas requiring further data quality assessment, particularly missing values and the suitability of data types for analysis.


## Importing Dataset

In [2]:

Project_Cleaning = pd.read_csv('Project-Cleaning.csv')
Project_Cleaning
# Display the first five rows
display(Project_Cleaning.head())

# Check the last five rows
display(Project_Cleaning.tail())

# Check the dataset dimensions
print("Dataset shape:", Project_Cleaning.shape)


# Display column names
print("Column names:", Project_Cleaning.columns.tolist())


FileNotFoundError: [Errno 2] No such file or directory: 'Project-Cleaning.csv'

In [ ]:
## Understanding the Data types
Project_Cleaning.dtypes

In [ ]:
##Check for missing values and count them
Project_Cleaning.isna().value_counts()

In [ ]:
Project_Cleaning.info()

In [ ]:


missing_summary = pd.DataFrame({
    "Missing Count": Project_Cleaning.isna().sum(),
    "Missing Percentage": (
        Project_Cleaning.isna().mean() * 100
    ).round(2)
})

missing_summary.sort_values(
    "Missing Count",
    ascending=False
)

In [ ]:
print("Duplicate rows:", Project_Cleaning.duplicated().sum())

print(
    "Duplicate OrderIDs:",
    Project_Cleaning["OrderID"].duplicated().sum()
)

In [ ]:
Project_Cleaning["Date"] = pd.to_datetime(
    Project_Cleaning["Date"],
    errors="coerce"
)

print("Missing dates:", Project_Cleaning["Date"].isna().sum())

### Missing Values and Duplicate Records

The dataset was assessed for missing values and duplicate records to evaluate its completeness and reliability. Missing values were identified in the **Date**, **CustomerID**, and **coupon code** columns. The Date column contained 529 missing values (44.1%), while CustomerID had 11 missing values. The coupon code column contained 309 missing values.

The missing dates occurred in a continuous block of records, and the missing CustomerID values overlapped with the final 11 records in this block. The coupon code values were missing across different order statuses. These missing values were retained during the initial analysis because their original values could not be established reliably.

The duplicate check identified **no duplicate rows**, indicating that each complete row was unique within the dataset. In addition, no duplicate OrderID values were found. These findings suggest that there were no exact duplicate records or repeated order identifiers, although further checks may still be needed to establish the overall accuracy of the data.


In [ ]:
Project_Cleaning[[
    "Quantity",
    "ItemsInCart",
    "UnitPrice",
    "TotalPrice"
]].describe()

In [ ]:
Project_Cleaning["calculated_total"] = (
    Project_Cleaning["Quantity"] * Project_Cleaning["UnitPrice"]
)

Project_Cleaning["price_difference"] = (
    Project_Cleaning["TotalPrice"] - Project_Cleaning["calculated_total"]
)

mismatch = Project_Cleaning[
    Project_Cleaning["price_difference"].abs() >= 0.01
]

print("Rows with differences of at least 0.01:",
      len(mismatch))

In [ ]:
product_summary = (
    Project_Cleaning.groupby("Product", dropna=False)
      .agg(
          total_revenue=("TotalPrice", "sum"),
          total_quantity=("Quantity", "sum")
      )
      .reset_index()
)

product_summary["average_revenue_per_unit"] = (
    product_summary["total_revenue"]
    / product_summary["total_quantity"]
)

product_summary = product_summary.sort_values(
    "total_revenue",
    ascending=False
)

product_summary

In [ ]:
print("Highest revenue:")
print(product_summary.iloc[0])

print("\nLowest revenue:")
print(product_summary.iloc[-1])

In [ ]:
product_summary[
    product_summary["Product"].isin(
        ["Chair", "Phone"]
    )
]

In [ ]:
product_summary.sort_values(
    "average_revenue_per_unit",
    ascending=False
)

### Product Sales Performance

Chair recorded the highest quantity sold, with 562 units, and generated total revenue of 195,620.10. Phone generated 151,722.40 from 411 units sold. Chair therefore had higher recorded revenue and quantity sold than Phone.

Tablet was identified as having the highest average revenue per unit in the earlier analysis. This measure describes revenue per unit and should not be interpreted as profit.

These findings describe the available dataset and do not establish the reasons for differences in product performance.

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=product_summary,
    x="Product",
    y="total_revenue"
)

plt.title("Total Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
quantity_summary = product_summary.sort_values(
    "total_quantity",
    ascending=False
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=quantity_summary,
    x="Product",
    y="total_quantity"
)

plt.title("Total Quantity Sold by Product")
plt.xlabel("Product")
plt.ylabel("Quantity Sold")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
status_summary = (
    Project_Cleaning.groupby("OrderStatus")["TotalPrice"]
      .agg(["sum", "count", "mean"])
      .sort_values("sum", ascending=False)
)

status_summary

In [ ]:
Project_Cleaning["PaymentMethod"].value_counts()

In [ ]:
Project_Cleaning["ReferralSource"].value_counts()

In [ ]:
pd.crosstab(
    Project_Cleaning["OrderStatus"],
    Project_Cleaning["CouponCode"].isna(),
    margins=True
)